# Growing Neural Cellular Automata (PyTorch)

This notebook trains a Neural Cellular Automata model to **grow a shape from a single seed cell**.

Based on the ["Growing Neural Cellular Automata"](http://distill.pub/2020/growing-ca) article, reimplemented in PyTorch.

**Key features:**
- Grows patterns from a single center seed point
- Alpha-based life mask (cells die without living neighbors)
- Regeneration training (patterns are damaged mid-training)
- Stochastic cell updates (configurable fire rate)
- Optional noise injection during evolution

In [ ]:
#@title Imports and Notebook Utilities
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output, display
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img


def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))


def parse_emoji(emoji_str):
    """Parse emoji string, handling both actual emoji and escape sequences.
    
    Accepts:
        - Actual emoji: "🦎"
        - Uppercase escape: "\\U0001F98E" or "\U0001F98E"
        - Lowercase escape: "\\u0001F98E" (for BMP characters)
        - Hex code: "1F98E" or "0x1F98E"
    
    Returns the actual emoji character.
    """
    s = emoji_str.strip()
    
    # If it's already a single character (actual emoji), return it
    if len(s) == 1:
        return s
    
    # Handle escape sequences that weren't interpreted
    if s.startswith('\\U') or s.startswith('\\u'):
        # e.g., "\\U0001F98E" -> extract hex part
        hex_str = s[2:]
        code_point = int(hex_str, 16)
        return chr(code_point)
    
    # Handle raw hex code (e.g., "1F98E" or "0x1F98E")
    if s.startswith('0x') or s.startswith('0X'):
        s = s[2:]
    
    # Try to parse as hex
    try:
        code_point = int(s, 16)
        return chr(code_point)
    except ValueError:
        pass
    
    # If string is longer than 1 char but not an escape, might be multi-codepoint emoji
    # Just return as-is
    return emoji_str


def load_emoji(emoji, max_size=40):
    """Load an emoji as an RGBA image from Google Noto Emoji.
    
    Args:
        emoji: Can be actual emoji ("🦎"), escape sequence ("\\U0001F98E"), 
               or hex code ("1F98E")
        max_size: Maximum size in pixels
    """
    # Parse the emoji string to get the actual character
    emoji_char = parse_emoji(emoji)
    
    # Get the Unicode code point
    code = hex(ord(emoji_char))[2:].lower()
    url = f'https://github.com/googlefonts/noto-emoji/blob/main/png/128/emoji_u{code}.png?raw=true'
    img = imread(url, max_size=max_size, mode='RGBA')
    # Premultiply RGB by alpha (standard for compositing)
    img[..., :3] *= img[..., 3:]
    return img


!nvidia-smi -L

In [ ]:
import torch
import torch.nn.functional as F

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title Load Target Image
from google.colab import files

#@markdown ### Target Selection
#@markdown Choose how to load the target image:
target_source = "emoji"  #@param ["emoji", "upload"]

#@markdown **Emoji settings** (only used if target_source="emoji"):
#@markdown Enter emoji directly (🐙) or as hex code (1F98E)
target_emoji = "1F98E"  #@param {type: "string"}
target_size = 40  #@param {type: "integer"}

#@markdown ### Padding
#@markdown Padding around the target image (organism grows into this space)
target_padding = 16  #@param {type: "integer"}

if target_source == "emoji":
    emoji_char = parse_emoji(target_emoji)
    print(f"Loading emoji: {emoji_char} (U+{ord(emoji_char):04X})")
    target_img = load_emoji(target_emoji, max_size=target_size)
else:
    print("Upload target image (must be RGBA/PNG):")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    target_img = imread(io.BytesIO(uploaded[filename]), max_size=target_size, mode='RGBA')
    # Premultiply RGB by alpha
    target_img[..., :3] *= target_img[..., 3:]

print(f"Target image shape: {target_img.shape}")

# Pad the target
p = target_padding
target_img_padded = np.pad(target_img, [(p, p), (p, p), (0, 0)], mode='constant')
print(f"Padded target shape: {target_img_padded.shape}")

# Convert to tensor [1, 4, H, W]
target_tensor = torch.tensor(target_img_padded).permute(2, 0, 1).unsqueeze(0)
h, w = target_img_padded.shape[:2]

# Display target
def to_rgb_display(img):
    """Convert RGBA (premultiplied) to RGB for display."""
    rgb, a = img[..., :3], img[..., 3:]
    return np.clip(1.0 - a + rgb, 0, 1)  # Blend with white background

print("\nTarget image (zoomed 4x):")
imshow(zoom(to_rgb_display(target_img_padded), 4), fmt='png')

In [ ]:
#@title GrowingNCA Architecture

#@markdown ### Model Architecture
channel_n = 12  #@param {type: "integer"}
fc_dim = 96  #@param {type: "integer"}
fire_rate = 0.5  #@param {type: "number"}


def depthwise_conv_zero_pad(x, filters):
    """Depthwise convolution with zero padding (not circular).
    
    Args:
        x: [b, ch, h, w]
        filters: [filter_n, 3, 3]
    Returns:
        [b, ch * filter_n, h, w]
    """
    b, ch, h, w = x.shape
    y = x.reshape(b * ch, 1, h, w)
    # Zero padding (default for conv2d)
    y = F.pad(y, [1, 1, 1, 1], mode='constant', value=0)
    y = F.conv2d(y, filters[:, None])
    return y.reshape(b, -1, h, w)


def merge_lap(z, chn):
    """Merge lap_x and lap_y into a single laplacian filter.
    
    Input: [b, 5 * chn, h, w] (ident, sobel_x, sobel_y, lap_x, lap_y per channel)
    Output: [b, 4 * chn, h, w] (ident, sobel_x, sobel_y, laplacian per channel)
    """
    b, c, h, w = z.shape
    z = torch.stack([
        z[:, 0::5],          # identity
        z[:, 1::5],          # sobel_x
        z[:, 2::5],          # sobel_y
        z[:, 3::5] + z[:, 4::5]  # lap_x + lap_y = laplacian
    ], dim=2)  # [b, chn, 4, h, w]
    return z.reshape(b, -1, h, w)  # [b, 4 * chn, h, w]


class GrowingNCA(torch.nn.Module):
    def __init__(self, chn=16, fc_dim=128, fire_rate=0.5):
        super().__init__()
        self.chn = chn
        self.fire_rate = fire_rate
        
        # Network: perception (4 filters per channel) -> fc1 -> relu -> fc2
        self.w1 = torch.nn.Conv2d(chn * 4, fc_dim, 1, bias=True)
        self.w2 = torch.nn.Conv2d(fc_dim, chn, 1, bias=False)
        
        # Initialize w2 to zeros (important for stable training start)
        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.5)
        torch.nn.init.zeros_(self.w2.weight)
        
        # Perception filters: identity, sobel_x, sobel_y, lap_x, lap_y
        # (same conventions as nca_training.ipynb - unnormalized filters)
        with torch.no_grad():
            ident = torch.tensor([[0.0, 0.0, 0.0], 
                                  [0.0, 1.0, 0.0], 
                                  [0.0, 0.0, 0.0]])
            sobel_x = torch.tensor([[-1.0, 0.0, 1.0], 
                                    [-2.0, 0.0, 2.0], 
                                    [-1.0, 0.0, 1.0]])
            lap_x = torch.tensor([[0.5, 0.0, 0.5], 
                                  [2.0, -6.0, 2.0], 
                                  [0.5, 0.0, 0.5]])
            self.register_buffer('filters', torch.stack([ident, sobel_x, sobel_x.T, lap_x, lap_x.T]))
    
    def get_living_mask(self, x):
        """Return mask of living cells (alpha > 0.1 in 3x3 neighborhood).
        
        Args:
            x: [b, chn, h, w] - state tensor (channel 3 is alpha)
        Returns:
            [b, 1, h, w] - boolean mask
        """
        alpha = x[:, 3:4, :, :]  # [b, 1, h, w]
        # Max pool to check if any neighbor has alpha > 0.1
        max_alpha = F.max_pool2d(alpha, kernel_size=3, stride=1, padding=1)
        return max_alpha > 0.1
    
    def perception(self, x):
        """Apply perception filters.
        
        Args:
            x: [b, chn, h, w]
        Returns:
            [b, chn * 4, h, w] - (identity, sobel_x, sobel_y, laplacian) per channel
        """
        z = depthwise_conv_zero_pad(x, self.filters)  # [b, 5 * chn, h, w]
        return merge_lap(z, self.chn)
    
    def forward(self, x, fire_rate=None, noise=None, noise_alpha=False):
        """Single CA update step.
        
        Args:
            x: [b, chn, h, w] - current state
            fire_rate: probability of cell update (None = use self.fire_rate)
            noise: optional noise level to add before perception
            noise_alpha: if True, add noise to alpha channel too
        Returns:
            [b, chn, h, w] - next state
        """
        if fire_rate is None:
            fire_rate = self.fire_rate
        
        # Pre-update life mask
        pre_life_mask = self.get_living_mask(x)
        
        # Add noise if specified
        if noise is not None and noise > 0:
            if noise_alpha:
                x = x + torch.randn_like(x) * noise
            else:
                # Only add noise to non-alpha channels
                noise_tensor = torch.randn_like(x) * noise
                noise_tensor[:, 3:4, :, :] = 0  # Zero out alpha noise
                x = x + noise_tensor
        
        # Perception
        z = self.perception(x)
        
        # Update rule: two-layer MLP
        dx = self.w2(torch.relu(self.w1(z)))
        
        # Stochastic update mask
        if fire_rate < 1.0:
            update_mask = (torch.rand(x.shape[0], 1, x.shape[2], x.shape[3]) <= fire_rate).float()
            x = x + dx * update_mask
        else:
            x = x + dx
        
        # Post-update life mask
        post_life_mask = self.get_living_mask(x)
        
        # Cell must be alive both before and after update
        life_mask = (pre_life_mask & post_life_mask).float()
        
        return x * life_mask
    
    def seed(self, n, h, w):
        """Create seed states (single center pixel initialized).
        
        Args:
            n: batch size
            h: height
            w: width
        Returns:
            [n, chn, h, w] - seed states
        """
        x = torch.zeros(n, self.chn, h, w)
        # Initialize center pixel: alpha=1, hidden channels=1
        x[:, 3:, h // 2, w // 2] = 1.0
        return x


def to_rgba(x):
    """Extract RGBA channels, clipping to [0, 1]."""
    return x[:, :4, :, :].clamp(0, 1)


def to_rgb(x):
    """Convert state to RGB for display (blend with white background)."""
    rgba = to_rgba(x)
    rgb, a = rgba[:, :3, :, :], rgba[:, 3:4, :, :]
    return 1.0 - a + rgb  # Blend with white


# Initialize model
model = GrowingNCA(chn=channel_n, fc_dim=fc_dim, fire_rate=fire_rate)
param_n = sum(p.numel() for p in model.parameters())
print(f'GrowingNCA with {channel_n} channels, fc_dim={fc_dim}, fire_rate={fire_rate}')
print(f'Parameter count: {param_n}')

In [ ]:
#@title Damage Utilities

def make_circle_masks(n, h, w, device='cuda'):
    """Create random circular damage masks.
    
    Args:
        n: number of masks
        h: height
        w: width
        device: torch device
    Returns:
        [n, 1, h, w] - binary masks (1 inside circle, 0 outside)
    """
    # Create coordinate grids
    y = torch.linspace(-1.0, 1.0, h, device=device)
    x = torch.linspace(-1.0, 1.0, w, device=device)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    
    # Random centers and radii
    center_x = torch.rand(n, 1, 1, device=device) * 1.0 - 0.5  # [-0.5, 0.5]
    center_y = torch.rand(n, 1, 1, device=device) * 1.0 - 0.5
    radius = torch.rand(n, 1, 1, device=device) * 0.3 + 0.1  # [0.1, 0.4]
    
    # Compute distance from center
    dx = (xx[None, :, :] - center_x) / radius
    dy = (yy[None, :, :] - center_y) / radius
    dist_sq = dx * dx + dy * dy
    
    # Mask: 1 inside circle, 0 outside
    masks = (dist_sq < 1.0).float()
    return masks.unsqueeze(1)  # [n, 1, h, w]


# Test the mask generation
test_masks = make_circle_masks(4, h, w)
print(f"Mask shape: {test_masks.shape}")

# Visualize
fig, axes = pl.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(test_masks[i, 0].cpu().numpy(), cmap='gray')
    ax.set_title(f'Mask {i+1}')
    ax.axis('off')
pl.suptitle('Example Damage Masks')
pl.tight_layout()
imshow(grab_plot())

In [ ]:
#@title Setup Training
import os
import glob
from google.colab import files

#@markdown ### Training Parameters
pool_size = 512  #@param {type: "integer"}
batch_size = 8  #@param {type: "integer"}
damage_n = 2  #@param {type: "integer"}

#@markdown ### Noise Settings
#@markdown Optional noise injection during evolution
runtime_noise = 0.01  #@param {type: "number"}
noise_affects_alpha = False  #@param {type: "boolean"}

print(f"Training settings:")
print(f"  Pool size: {pool_size}")
print(f"  Batch size: {batch_size}")
print(f"  Damage per batch: {damage_n}")
print(f"  Runtime noise: {runtime_noise}")
print(f"  Noise affects alpha: {noise_affects_alpha}")

# Check for existing weight files
weight_files = glob.glob('growing_nca_*.pt') + glob.glob('checkpoint_*.pt') + glob.glob('weights.pt')
weight_files = list(set(weight_files))

state_dict = None

if weight_files:
    print(f"\nFound {len(weight_files)} weight file(s):")
    for i, f in enumerate(weight_files):
        file_size_kb = os.path.getsize(f) / 1024
        print(f"  [{i}] {f} ({file_size_kb:.1f} KB)")
    print("\nOptions:")
    print("  [number] - Load weights")
    print("  u - Upload weights file")
    print("  Enter - Fresh start")
    choice = input("Choice: ").strip()

    if choice == 'u':
        print("Please upload your weights file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        state_dict = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(weight_files):
        filename = weight_files[int(choice)]
        print(f'Loading weights from: "{filename}"')
        state_dict = torch.load(filename)
else:
    print("\nNo weight files found in Colab storage.")
    upload_choice = input("Upload a weights file? (y/n, default=n): ").strip().lower()
    if upload_choice == 'y':
        print("Please upload your weights file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        state_dict = torch.load(filename)

# Load weights into model
if state_dict is not None:
    if 'model_state_dict' in state_dict:
        model_weights = state_dict['model_state_dict']
        start_iter = state_dict.get('iteration', 0) + 1
        loss_log = state_dict.get('loss_log', [])
    else:
        model_weights = {k: v for k, v in state_dict.items() if not k.startswith('_')}
        start_iter = 0
        loss_log = []
    model.load_state_dict(model_weights)
    print(f"Loaded weights (starting from iteration {start_iter})")
else:
    start_iter = 0
    loss_log = []
    print("Starting fresh training")

# Create seed state
seed = model.seed(1, h, w)
print(f"\nSeed shape: {seed.shape}")

# Initialize pool with seeds
with torch.no_grad():
    pool = seed.repeat(pool_size, 1, 1, 1)
print(f"Pool shape: {pool.shape}")

# Loss function: MSE on RGBA
def loss_f(x):
    """Compute per-sample MSE loss on RGBA channels.
    
    Args:
        x: [b, chn, h, w]
    Returns:
        [b] - per-sample loss
    """
    rgba = to_rgba(x)
    target = target_tensor.expand(x.shape[0], -1, -1, -1)
    return ((rgba - target) ** 2).mean(dim=(1, 2, 3))

# Optimizer with adaptive LR
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.5, patience=500,
    threshold=0.01, threshold_mode='rel', min_lr=1e-5
)

print(f"\nReady to train!")

In [ ]:
#@title Training Loop {vertical-output: true}

num_iterations = 8000  #@param {type: "integer"}

try:
    for i in range(start_iter, start_iter + num_iterations):
        # Sample batch from pool
        batch_idx = np.random.choice(pool_size, batch_size, replace=False)
        
        with torch.no_grad():
            x = pool[batch_idx].clone()
            
            # Sort by loss (highest first) - replace worst with fresh seed
            losses = loss_f(x)
            loss_rank = losses.argsort(descending=True)
            x = x[loss_rank]
            batch_idx = batch_idx[loss_rank.cpu().numpy()]
            
            # Replace highest-loss sample with fresh seed
            x[0] = seed[0]
            
            # Damage last damage_n samples with circle masks
            if damage_n > 0:
                damage_masks = make_circle_masks(damage_n, h, w)
                x[-damage_n:] = x[-damage_n:] * (1.0 - damage_masks)
        
        # Run forward steps
        step_n = np.random.randint(32, 96)
        for _ in range(step_n):
            x = model(x, noise=runtime_noise, noise_alpha=noise_affects_alpha)
        
        # Compute loss (mean over batch)
        loss = loss_f(x).mean()
        
        # Backward and optimize
        opt.zero_grad()
        loss.backward()
        
        # Normalize gradients (from original paper)
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data = p.grad.data / (p.grad.data.norm() + 1e-8)
        
        opt.step()
        lr_sched.step(loss)
        
        # Update pool
        with torch.no_grad():
            pool[batch_idx] = x.detach()
        
        loss_log.append(loss.item())
        
        # Display progress
        if i % 10 == 0:
            lr = opt.param_groups[0]['lr']
            display(Markdown(f"iter: {i}, loss: {loss.item():.4f}, log10(loss): {np.log10(loss.item()):.2f}, lr: {lr:.2e}"), display_id='stats')
        
        # Visualize
        if i % 25 == 0:
            with torch.no_grad():
                # Show loss plot and current batch
                pl.figure(figsize=(14, 4))
                
                pl.subplot(1, 3, 1)
                pl.plot(np.log10(loss_log), '.', alpha=0.1)
                pl.title('log10(Loss)')
                pl.xlabel('Iteration')
                
                # Show current batch (before/after would require storing x0)
                pl.subplot(1, 3, 2)
                rgb = to_rgb(x)
                imgs = rgb.permute(0, 2, 3, 1).cpu().numpy()
                vis = np.hstack(imgs[:4])  # Show first 4
                pl.imshow(vis)
                pl.title('Current batch (first 4)')
                pl.axis('off')
                
                # Show target
                pl.subplot(1, 3, 3)
                pl.imshow(to_rgb_display(target_img_padded))
                pl.title('Target')
                pl.axis('off')
                
                pl.tight_layout()
                imshow(grab_plot(), id='progress')
        
        # Save checkpoint
        if i % 1000 == 0 and i > start_iter:
            checkpoint = {
                'iteration': i,
                'model_state_dict': model.state_dict(),
                'loss_log': loss_log,
                'channel_n': channel_n,
                'fc_dim': fc_dim,
                'fire_rate': fire_rate,
                'runtime_noise': runtime_noise,
                'noise_affects_alpha': noise_affects_alpha,
            }
            torch.save(checkpoint, f'growing_nca_iter_{i}.pt')
            print(f"\nCheckpoint saved at iteration {i}")

except KeyboardInterrupt:
    print('\n\nTraining interrupted by user!')
    print(f'Saving checkpoint at iteration {i}...')
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'loss_log': loss_log,
        'channel_n': channel_n,
        'fc_dim': fc_dim,
        'fire_rate': fire_rate,
        'runtime_noise': runtime_noise,
        'noise_affects_alpha': noise_affects_alpha,
    }
    torch.save(checkpoint, f'growing_nca_interrupted_iter_{i}.pt')
    print(f'Checkpoint saved!')

print(f'\nTraining completed at iteration {i}')

In [ ]:
#@title Test Model - Growth and Regeneration

print("Testing growth from seed...")

with torch.no_grad():
    # Grow from seed
    x = model.seed(1, h, w)
    
    growth_frames = [to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()]
    
    for step in range(200):
        x = model(x, noise=runtime_noise, noise_alpha=noise_affects_alpha)
        if step % 4 == 0:
            growth_frames.append(to_rgb(x)[0].permute(1, 2, 0).cpu().numpy())
    
    # Show growth sequence
    fig, axes = pl.subplots(2, 6, figsize=(15, 5))
    indices = [0, 5, 10, 20, 30, 50]
    for i, idx in enumerate(indices):
        axes[0, i].imshow(growth_frames[idx])
        axes[0, i].set_title(f'Step {idx * 4}')
        axes[0, i].axis('off')
    
    # Test regeneration
    print("\nTesting regeneration...")
    
    # Damage the grown organism
    x_damaged = x.clone()
    damage_mask = make_circle_masks(1, h, w)
    x_damaged = x_damaged * (1.0 - damage_mask)
    
    regen_frames = [to_rgb(x_damaged)[0].permute(1, 2, 0).cpu().numpy()]
    
    for step in range(200):
        x_damaged = model(x_damaged, noise=runtime_noise, noise_alpha=noise_affects_alpha)
        if step % 4 == 0:
            regen_frames.append(to_rgb(x_damaged)[0].permute(1, 2, 0).cpu().numpy())
    
    for i, idx in enumerate(indices):
        axes[1, i].imshow(regen_frames[idx])
        axes[1, i].set_title(f'Regen {idx * 4}')
        axes[1, i].axis('off')
    
    axes[0, 0].set_ylabel('Growth', fontsize=12)
    axes[1, 0].set_ylabel('Regeneration', fontsize=12)
    
    pl.tight_layout()
    imshow(grab_plot())

print("\nCompare with target:")
fig, axes = pl.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_rgb_display(target_img_padded))
axes[0].set_title('Target')
axes[0].axis('off')

axes[1].imshow(growth_frames[-1])
axes[1].set_title('Grown')
axes[1].axis('off')

axes[2].imshow(regen_frames[-1])
axes[2].set_title('Regenerated')
axes[2].axis('off')

pl.tight_layout()
imshow(grab_plot())

In [ ]:
#@title Create Growth Video

video_steps = 300  #@param {type: "integer"}
video_filename = "growth.mp4"  #@param {type: "string"}

print(f"Creating growth video ({video_steps} steps)...")

with torch.no_grad():
    x = model.seed(1, h, w)
    
    with VideoWriter(video_filename, fps=30.0) as vid:
        for step in range(video_steps):
            rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
            vid.add(zoom(rgb, 4))
            x = model(x, noise=runtime_noise, noise_alpha=noise_affects_alpha)

print(f"Saved: {video_filename}")
mvp.ipython_display(video_filename)

In [ ]:
#@title Create Regeneration Video

grow_steps = 150  #@param {type: "integer"}
regen_steps = 200  #@param {type: "integer"}
video_filename = "regeneration.mp4"  #@param {type: "string"}

print(f"Creating regeneration video...")

with torch.no_grad():
    x = model.seed(1, h, w)
    
    with VideoWriter(video_filename, fps=30.0) as vid:
        # Grow
        for step in range(grow_steps):
            rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
            vid.add(zoom(rgb, 4))
            x = model(x, noise=runtime_noise, noise_alpha=noise_affects_alpha)
        
        # Damage
        damage_mask = make_circle_masks(1, h, w)
        x = x * (1.0 - damage_mask)
        
        # Pause on damage frame
        rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
        for _ in range(30):
            vid.add(zoom(rgb, 4))
        
        # Regenerate
        for step in range(regen_steps):
            rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
            vid.add(zoom(rgb, 4))
            x = model(x, noise=runtime_noise, noise_alpha=noise_affects_alpha)

print(f"Saved: {video_filename}")
mvp.ipython_display(video_filename)

In [ ]:
#@title Save Model Weights

weights_file = 'weights.pt'  #@param {type: "string"}

# Include training params
state_dict = model.state_dict()
state_dict['_training_params'] = {
    'model_type': 'growing_nca',
    'channel_n': channel_n,
    'fc_dim': fc_dim,
    'fire_rate': fire_rate,
    'runtime_noise': runtime_noise,
    'noise_affects_alpha': noise_affects_alpha,
    'target_h': h,
    'target_w': w,
    'target_padding': target_padding,
}

torch.save(state_dict, weights_file)
print(f"Saved: {weights_file}")
print(f"  Model type: growing_nca")
print(f"  Channels: {channel_n}")
print(f"  FC dim: {fc_dim}")
print(f"  Fire rate: {fire_rate}")
print(f"  Runtime noise: {runtime_noise}")
print(f"  Noise affects alpha: {noise_affects_alpha}")
print(f"  Target size: {h}x{w} (padding={target_padding})")

# Download
from google.colab import files
files.download(weights_file)